In [ ]:
import sys
sys.path.append("./../")
import numpy as np
import tensorflow as tf

In [ ]:
from pycomet.networks import EncodeProcessDecode

In [ ]:
edge_index = [[0,1],[0,2],[0,3],[0,4],[1,2],[3,4]]
edge_index = np.array(edge_index).T
x = [np.random.rand(20,5,2), np.random.rand(20,6,2)]
x = tf.nest.map_structure(tf.convert_to_tensor, x)

In [ ]:
dim = {
	"inp": [(5,2)],  # Input node attributes
	"out": [(5,1)]   # Output node attributes
}

# Layers
# -------------------------------------
latent_dim = 8

common = {
  'kernel_initializer': 'GlorotNormal',
  'regularizer': {
    'name': 'l1+l2',
    'l1': 1e-4,
    'l2': 1e-4
  },
  'dropout': {'rate': 1e-3},
  'layer_norm': True,
  'trainable': True
}


lays_propty = {

  'edge_encoder': {
    'lay_id': 'EdgeEncoder',
    'kwargs': {
      'layer_size': [2, 32, 32, latent_dim],
      'activation': ['sigmoid']*2 + ['linear'],
      "static_edge_index": edge_index,
      "static_edge_attr": None,
      **common
    }
  },

  'node_encoder': {
    'lay_id': 'FFBlock',
    'kwargs': {
      'layer_size': [2, 32, 32, latent_dim],
      'activation': ['sigmoid']*2 + ['linear'],
      **common
    }
  },

  'graphnet': [{
    'lay_id': 'GraphNet',
    'kwargs': {
      'layer_size': {
        "node": [latent_dim, 32, 32, latent_dim],
        "edge": [latent_dim, 32, 32, latent_dim]
      },
      'activation': {
        "node": ['tanh']*2 + ['linear'],
        "edge": ['tanh']*2 + ['linear']
      },
      "static_edge_index": edge_index,
      **common
    }
  } for _ in range(2)],

  'node_decoder': {
    'lay_id': 'FFBlock',
    'kwargs': {
      'layer_size': [latent_dim, 32, 32, 1],
      'activation': ['sigmoid']*2 + ['linear'],
      **common
    }
  }

}


In [ ]:
model = EncodeProcessDecode(
    dim=dim,
    lays_propty=lays_propty
)

In [ ]:
y = model.forward(x, training=False)

In [ ]:
y.shape